# 1 - Sequence to Sequence Learning with Neural Networks
## 任务五：Seq2Seq 德语→英语翻译模型

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import random
import numpy as np
from torch.nn.utils.rnn import pad_sequence

# 固定随机种子
seed = 1234
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.backends.cudnn.deterministic = True

In [ ]:
# 德语-英语数据集
data = [
    ("Ich liebe dich", "I love you"),
    ("Guten Morgen", "Good morning"),
    ("Wie geht es dir", "How are you"),
    ("Das ist gut", "That is good"),
    ("Ich bin Student", "I am a student"),
    ("Sie ist nett", "She is nice"),
    ("Er lernt Deutsch", "He learns German"),
    ("Ich habe Hunger", "I am hungry")
]

In [ ]:
# 分词
def tokenize_de(text):
    return text.split()[::-1]

def tokenize_en(text):
    return text.split()

In [ ]:
# 词典类
class Vocab:
    def __init__(self, tokens):
        self.stoi = {"<sos>":0, "<eos>":1, "<pad>":2, "<unk>":3}
        self.itos = {0:"<sos>",1:"<eos>",2:"<pad>",3:"<unk>"}
        self.build_vocab(tokens)
    
    def build_vocab(self, tokens):
        for tok in tokens:
            if tok not in self.stoi:
                self.stoi[tok] = len(self.stoi)
                self.itos[len(self.itos)] = tok
    
    def encode(self, seq):
        return [self.stoi.get(tok,3) for tok in seq] + [1]

In [ ]:
# 构建词典
all_de = []
all_en = []
for de, en in data:
    all_de += tokenize_de(de)
    all_en += tokenize_en(en)

de_vocab = Vocab(all_de)
en_vocab = Vocab(all_en)
PAD_IDX = de_vocab.stoi["<pad>"]

# 模型 Model
## Encoder / Decoder / Seq2Seq

In [ ]:
# Encoder
class Encoder(nn.Module):
    def __init__(self, input_dim, emb_dim, hid_dim, n_layers, dropout):
        super().__init__()
        self.embedding = nn.Embedding(input_dim, emb_dim)
        self.lstm = nn.LSTM(emb_dim, hid_dim, n_layers, dropout=dropout)
        self.dropout = nn.Dropout(dropout)
    def forward(self, src):
        embedded = self.dropout(self.embedding(src))
        outputs, (hidden, cell) = self.lstm(embedded)
        return hidden, cell

# Decoder
class Decoder(nn.Module):
    def __init__(self, output_dim, emb_dim, hid_dim, n_layers, dropout):
        super().__init__()
        self.embedding = nn.Embedding(output_dim, emb_dim)
        self.lstm = nn.LSTM(emb_dim, hid_dim, n_layers, dropout=dropout)
        self.fc = nn.Linear(hid_dim, output_dim)
        self.dropout = nn.Dropout(dropout)
    def forward(self, trg, hidden, cell):
        trg = trg.unsqueeze(0)
        embedded = self.dropout(self.embedding(trg))
        output, (hidden, cell) = self.lstm(embedded, (hidden, cell))
        prediction = self.fc(output.squeeze(0))
        return prediction, hidden, cell

# Seq2Seq
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device
    def forward(self, src, trg, teacher_forcing_ratio=0.5):
        trg_len, batch_size = trg.shape
        trg_vocab_size = self.decoder.fc.out_features
        outputs = torch.zeros(trg_len, batch_size, trg_vocab_size).to(self.device)
        hidden, cell = self.encoder(src)
        input = trg[0,:]
        for t in range(1, trg_len):
            output, hidden, cell = self.decoder(input, hidden, cell)
            outputs[t] = output
            top1 = output.argmax(1)
            input = trg[t] if random.random() < teacher_forcing_ratio else top1
        return outputs

In [ ]:
# 超参数
INPUT_DIM = len(de_vocab.stoi)
OUTPUT_DIM = len(en_vocab.stoi)
ENC_EMB_DIM = 64
DEC_EMB_DIM = 64
HID_DIM = 128
N_LAYERS = 2
ENC_DROPOUT = 0.3
DEC_DROPOUT = 0.3

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

enc = Encoder(INPUT_DIM, ENC_EMB_DIM, HID_DIM, N_LAYERS, ENC_DROPOUT)
dec = Decoder(OUTPUT_DIM, DEC_EMB_DIM, HID_DIM, N_LAYERS, DEC_DROPOUT)
model = Seq2Seq(enc, dec, device).to(device)

optimizer = optim.Adam(model.parameters())
criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)

In [ ]:
# 训练一步
def train_step(model, batch, optimizer, criterion, clip=1.0):
    model.train()
    src, trg = batch
    src = src.to(device)
    trg = trg.to(device)
    optimizer.zero_grad()
    output = model(src, trg)
    output_dim = output.shape[-1]
    output = output[1:].reshape(-1, output_dim)
    trg = trg[1:].reshape(-1)
    loss = criterion(output, trg)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
    optimizer.step()
    return loss.item()

# 构造批次
def make_batch(data):
    src_batch, trg_batch = [], []
    for de, en in data:
        src = de_vocab.encode(tokenize_de(de))
        trg = en_vocab.encode(tokenize_en(en))
        src_batch.append(torch.tensor(src))
        trg_batch.append(torch.tensor(trg))
    src_padded = pad_sequence(src_batch, padding_value=PAD_IDX)
    trg_padded = pad_sequence(trg_batch, padding_value=PAD_IDX)
    return src_padded, trg_padded

batch = make_batch(data)

In [ ]:
# 训练
EPOCHS = 200
print("开始训练...")
for epoch in range(EPOCHS):
    loss = train_step(model, batch, optimizer, criterion)
    if epoch % 20 == 0:
        print(f"Epoch {epoch:03d} | Loss: {loss:.4f}")
print("训练完成！")

In [ ]:
# 翻译测试
def translate_sentence(sentence, model, de_vocab, en_vocab, device, max_len=10):
    model.eval()
    tokens = tokenize_de(sentence)
    src_indexes = de_vocab.encode(tokens)
    src_tensor = torch.tensor(src_indexes).unsqueeze(1).to(device)
    with torch.no_grad():
        hidden, cell = model.encoder(src_tensor)
    trg_indexes = [en_vocab.stoi["<sos>"]]
    for _ in range(max_len):
        trg_tensor = torch.tensor([trg_indexes[-1]]).to(device)
        with torch.no_grad():
            output, hidden, cell = self.decoder(trg_tensor, hidden, cell)
        pred_token = output.argmax(1).item()
        trg_indexes.append(pred_token)
        if pred_token == en_vocab.stoi["<eos>"]:
            break
    trg_tokens = [en_vocab.itos[i] for i in trg_indexes]
    return trg_tokens[1:-1]

test_sentence = "Ich liebe dich"
translation = translate_sentence(test_sentence, model, de_vocab, en_vocab, device)
print("\n德语:", test_sentence)
print("翻译:", " ".join(translation))